# Validacion y metrica de negocio de mi proyecto (M5, CA_1)

Modelo final: **gradient boosting** (decision de la sesion pasada). Segundo mejor: **red neuronal**. Se comparan con validacion cruzada de 5 pliegues, con dos esquemas: **temporal** (el que corresponde, porque el modelo predice el futuro a partir del pasado) y **al azar** (para comparar).

Para que la validacion cruzada sea viable se usa un subconjunto: 150 items elegidos al azar (random_state=42), con sus ventas de los ultimos 730 dias.

**Costo del error (supuesto, ajustable):** subestimar la demanda es mas caro que sobreestimarla. Si el pronostico queda corto, se agota el producto y se pierde la venta (se pierde el margen); si queda alto, sobra inventario (merma y capital inmovilizado). Se supone que subestimar una unidad cuesta el doble que sobreestimarla: costo = 2 x unidades de menos + 1 x unidades de mas.

In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import make_scorer
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_validate
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

from pipeline_proyecto import CATEGORICAS, NUMERICAS, OBJETIVO, YA_LISTAS, cargar_completo, construir_preparacion

COSTO_SUBESTIMAR = 2.0   # por unidad que se pronostica de menos
COSTO_SOBREESTIMAR = 1.0  # por unidad que se pronostica de mas

datos = cargar_completo()
items = pd.Series(datos["item_id"].unique()).sample(150, random_state=42)
datos = datos[datos["item_id"].isin(items) & (datos["d_num"] > datos["d_num"].max() - 730)]
datos = datos.sort_values(["d_num", "item_id"]).reset_index(drop=True)
X = datos[NUMERICAS + CATEGORICAS + YA_LISTAS]
y = datos[OBJETIVO]
print("Filas:", len(datos), " dias:", datos["d_num"].min(), "a", datos["d_num"].max())

Filas: 107386  dias: 1212 a 1941


In [2]:
def costo_negocio(real, estimada):
    """Costo medio por fila: 2 por cada unidad de menos, 1 por cada unidad de mas."""
    falta = np.clip(np.asarray(real) - np.asarray(estimada), 0, None)
    sobra = np.clip(np.asarray(estimada) - np.asarray(real), 0, None)
    return float(np.mean(COSTO_SUBESTIMAR * falta + COSTO_SOBREESTIMAR * sobra))


def subestimados_20(real, estimada):
    real, estimada = np.asarray(real), np.asarray(estimada)
    return float(np.mean(real > 1.2 * estimada + 1e-9))


puntajes = {
    "mae": "neg_mean_absolute_error",
    "costo": make_scorer(costo_negocio, greater_is_better=False),
}
modelos = {
    "gradient boosting (final)": lambda: GradientBoostingRegressor(random_state=42),
    "red neuronal (segundo)": lambda: MLPRegressor(hidden_layer_sizes=(64,), max_iter=1000, random_state=42),
}
esquemas = {
    "temporal": TimeSeriesSplit(n_splits=5),
    "al azar": KFold(n_splits=5, shuffle=True, random_state=42),
}

In [3]:
filas = []
for esquema, cv in esquemas.items():
    for modelo, crear in modelos.items():
        pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", crear())])
        r = cross_validate(pipeline, X, y, cv=cv, scoring=puntajes)
        mae, costo = -r["test_mae"], -r["test_costo"]
        filas.append({"esquema": esquema, "modelo": modelo,
                      "mae_promedio": mae.mean(), "mae_desviacion": mae.std(),
                      "costo_promedio": costo.mean(), "costo_desviacion": costo.std(),
                      **{f"mae_pliegue_{i + 1}": v for i, v in enumerate(mae)}})
        print(f"{esquema:9s} {modelo:26s} MAE {mae.mean():.4f} +- {mae.std():.4f}   costo {costo.mean():.4f} +- {costo.std():.4f}   pliegues {np.round(mae, 3)}")
tabla = pd.DataFrame(filas).round(4)
tabla.to_csv("validacion_proyecto.csv", index=False)
tabla

temporal  gradient boosting (final)  MAE 1.0138 +- 0.0249   costo 1.5224 +- 0.0384   pliegues [1.054 1.001 1.015 0.978 1.02 ]


temporal  red neuronal (segundo)     MAE 1.0672 +- 0.0430   costo 1.6779 +- 0.1221   pliegues [1.12  1.063 1.108 1.004 1.04 ]


al azar   gradient boosting (final)  MAE 1.0192 +- 0.0091   costo 1.5287 +- 0.0181   pliegues [1.009 1.018 1.036 1.016 1.018]


al azar   red neuronal (segundo)     MAE 1.0410 +- 0.0132   costo 1.5633 +- 0.0145   pliegues [1.019 1.059 1.037 1.044 1.047]


,esquema,modelo,mae_promedio,mae_desviacion,costo_promedio,costo_desviacion,mae_pliegue_1,mae_pliegue_2,mae_pliegue_3,mae_pliegue_4,mae_pliegue_5
0,temporal,gradient boosting (final),1.0138,0.0249,1.5224,0.0384,1.0539,1.0011,1.0155,0.9781,1.0201
1,temporal,red neuronal (segundo),1.0672,0.0430,1.6779,0.1221,1.1203,1.0631,1.1082,1.0037,1.0404
2,al azar,gradient boosting (final),1.0192,0.0091,1.5287,0.0181,1.0085,1.0180,1.0361,1.0155,1.0178
3,al azar,red neuronal (segundo),1.0410,0.0132,1.5633,0.0145,1.0188,1.0588,1.0365,1.0440,1.0469


## Comparar el modelo final con el segundo, pliegue por pliegue

In [4]:
pares = tabla.pivot(index="esquema", columns="modelo", values=["mae_promedio", "mae_desviacion", "costo_promedio"])
pares

mae_promedio                         \
modelo   gradient boosting (final) red neuronal (segundo)   
esquema                                                     
al azar                     1.0192                 1.0410   
temporal                    1.0138                 1.0672   

                    mae_desviacion                         \
modelo   gradient boosting (final) red neuronal (segundo)   
esquema                                                     
al azar                     0.0091                 0.0132   
temporal                    0.0249                 0.0430   

                    costo_promedio                         
modelo   gradient boosting (final) red neuronal (segundo)  
esquema                                                    
al azar                     1.5287                 1.5633  
temporal                    1.5224                 1.6779